In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# WEEK 8 — FUNCTION 4 (PROMPT/DECODING-AWARE v3 — GREEDY BEST-BY-SCORE)
# Update requested:
#  - ALWAYS output the best-by-score x_next (no sampling)
#  - Print x_next in 6 decimals
#  - Full runnable code (single script)
# ============================================================

# ============================================================
# 1) Input data (Function 4)
# ============================================================
X_train = np.array([
    [0.89698105, 0.72562797, 0.17540431, 0.70169437],
    [0.8893564 , 0.49958786, 0.53926886, 0.50878344],
    [0.25094624, 0.03369313, 0.14538002, 0.49493242],
    [0.34696206, 0.0062504 , 0.76056361, 0.61302356],
    [0.12487118, 0.12977019, 0.38440048, 0.2870761 ],
    [0.80130271, 0.50023109, 0.70664456, 0.19510284],
    [0.24770826, 0.06044543, 0.04218635, 0.44132425],
    [0.74670224, 0.7570915 , 0.36935306, 0.20656628],
    [0.40066503, 0.07257425, 0.88676825, 0.24384229],
    [0.6260706 , 0.58675126, 0.43880578, 0.77885769],
    [0.95713529, 0.59764438, 0.76611385, 0.77620991],
    [0.73281243, 0.14524998, 0.47681272, 0.13336573],
    [0.65511548, 0.07239183, 0.68715175, 0.08151656],
    [0.21973443, 0.83203134, 0.48286416, 0.08256923],
    [0.48859419, 0.2119651 , 0.93917791, 0.37619173],
    [0.16713049, 0.87655456, 0.21723954, 0.95980098],
    [0.21691119, 0.16608583, 0.24137226, 0.77006248],
    [0.38748784, 0.80453226, 0.75179548, 0.72382744],
    [0.98562189, 0.66693268, 0.15678328, 0.8565348 ],
    [0.03782483, 0.66485335, 0.16198218, 0.25392378],
    [0.68348638, 0.9027701 , 0.33541983, 0.99948256],
    [0.17034731, 0.75695908, 0.27652049, 0.5312315 ],
    [0.85965692, 0.91959232, 0.20613873, 0.09779683],
    [0.28213837, 0.50598691, 0.53053084, 0.09630162],
    [0.32607578, 0.4723669 , 0.453192  , 0.10588734],
    [0.94838936, 0.89451301, 0.85163782, 0.55219629],
    [0.66495539, 0.04656628, 0.11677747, 0.79371778],
    [0.57776561, 0.42877174, 0.42582587, 0.24900741],
    [0.73861301, 0.48210263, 0.70936644, 0.50397001],
    [0.8548108 , 0.49396462, 0.73530997, 0.80809201],
    [1.085621  , 1.019592  , 1.039177  , 1.099482  ],
    [1.00000e-06, 1.00000e-06, 1.24558e-01, 1.00000e-06],
    [0.866175  , 0.601115  , 0.708072  , 0.020585  ],
    [0.145904  , 0.536548  , 0.6014    , 0.01905   ],
    [0.356293  , 0.442523  , 0.13052   , 0.242559  ],
    [0.061431  , 0.381247  , 0.983792  , 0.705575  ],
    [0.497045, 0.450388, 0.380113, 0.297612]
], dtype=float)

y_train = np.array([
    -22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
    -15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
    -27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
    -26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
    -29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
    -32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284 ,
    -67.60493430274798, -22.782193418373407, -22.194212794446454,
    -13.363105653346768,  -5.926020577803715, -23.786955955997737,
    -2.5615259470796796
], dtype=float)

assert len(X_train) == len(y_train), "X_train and y_train length mismatch"

# ============================================================
# 2) Current best (maximisation)
# ============================================================
current_best_idx = int(np.argmax(y_train))
current_best_x = X_train[current_best_idx]
current_best_y = float(y_train[current_best_idx])

print("Current best index:", current_best_idx)
print("Current best X:", current_best_x)
print("Current best y:", current_best_y)

# ============================================================
# 3) Scale X to [0,1] per feature + standardise y
# ============================================================
X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)
X_scaled = (X_train - X_min) / (X_max - X_min + 1e-12)

y_mean = y_train.mean()
y_std = y_train.std() + 1e-12
y_scaled = (y_train - y_mean) / y_std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor_all = torch.tensor(y_scaled, dtype=torch.float32, device=device).unsqueeze(-1)

torch.manual_seed(42)
np.random.seed(42)
rng = np.random.default_rng(42)

# ============================================================
# 4) MLP surrogate
# ============================================================
class MLP(nn.Module):
    def __init__(self, input_dim=4, hidden=(64, 64), p_dropout=0.1):
        super().__init__()
        h1, h2 = hidden
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h2, 1),
        )

    def forward(self, x):
        return self.net(x)

def train_model(model, X, y, epochs=900, lr=1e-3, weight_decay=1e-5):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
    return float(loss.item())

# ============================================================
# 5) CV + Random Search tuning
# ============================================================
def kfold_indices(n, k=4, seed=123):
    rr = np.random.default_rng(seed)
    idx = np.arange(n)
    rr.shuffle(idx)
    return np.array_split(idx, k)

def cv_mse_for_config(cfg, X_tensor, y_tensor, k=4, seed=123):
    folds = kfold_indices(len(X_tensor), k=k, seed=seed)
    mses = []
    for i in range(k):
        val_idx = folds[i]
        tr_idx = np.concatenate([folds[j] for j in range(k) if j != i])

        X_tr, y_tr = X_tensor[tr_idx], y_tensor[tr_idx]
        X_val, y_val = X_tensor[val_idx], y_tensor[val_idx]

        torch.manual_seed(1000 + i)
        model = MLP(input_dim=4, hidden=cfg["hidden"], p_dropout=cfg["dropout"]).to(device)
        train_model(model, X_tr, y_tr, epochs=cfg["epochs"], lr=cfg["lr"], weight_decay=cfg["weight_decay"])

        model.eval()
        with torch.no_grad():
            pred = model(X_val)
            mse = nn.MSELoss()(pred, y_val).item()
        mses.append(mse)

    return float(np.mean(mses))

def sample_config(rr):
    hidden_choices = [(32, 32), (64, 32), (64, 64), (128, 64)]
    dropout_choices = [0.0, 0.05, 0.1, 0.2]
    lr_choices = [1e-4, 3e-4, 1e-3, 3e-3]
    wd_choices = [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
    epochs_choices = [700, 900, 1200]
    n_ens_choices = [7, 9]
    xi_choices = [0.0, 0.005, 0.01]

    return {
        "hidden": hidden_choices[rr.integers(0, len(hidden_choices))],
        "dropout": float(dropout_choices[rr.integers(0, len(dropout_choices))]),
        "lr": float(lr_choices[rr.integers(0, len(lr_choices))]),
        "weight_decay": float(wd_choices[rr.integers(0, len(wd_choices))]),
        "epochs": int(epochs_choices[rr.integers(0, len(epochs_choices))]),
        "n_ensemble": int(n_ens_choices[rr.integers(0, len(n_ens_choices))]),
        "xi": float(xi_choices[rr.integers(0, len(xi_choices))]),
    }

N_TRIALS = 35
K_FOLDS = 4

best_cfg = None
best_cv = float("inf")

print("\n=== Hyperparameter tuning (random search + {}-fold CV) ===".format(K_FOLDS))
for t in range(N_TRIALS):
    cfg = sample_config(rng)
    cv = cv_mse_for_config(cfg, X_tensor_all, y_tensor_all, k=K_FOLDS, seed=123)
    if cv < best_cv:
        best_cv = cv
        best_cfg = cfg
    print(f"trial {t+1:02d}/{N_TRIALS}  cv_mse={cv:.6f}  cfg={cfg}")

print("\n=== Best tuned configuration ===")
print("Best CV-MSE (scaled y):", best_cv)
print("Best cfg:", best_cfg)

# ============================================================
# 6) Train tuned BOOTSTRAPPED ensemble
# ============================================================
def train_ensemble(cfg, X_tensor, y_tensor, rr):
    ensemble = []
    n = len(X_tensor)
    for m in range(cfg["n_ensemble"]):
        boot_idx = rr.integers(0, n, size=n)
        Xb = X_tensor[boot_idx]
        yb = y_tensor[boot_idx]

        torch.manual_seed(500 + m)
        model = MLP(input_dim=4, hidden=cfg["hidden"], p_dropout=cfg["dropout"]).to(device)
        train_model(model, Xb, yb, epochs=cfg["epochs"], lr=cfg["lr"], weight_decay=cfg["weight_decay"])
        ensemble.append(model)
    return ensemble

ensemble = train_ensemble(best_cfg, X_tensor_all, y_tensor_all, rng)

# ============================================================
# 7) Prediction (original y units) + EI/PI
# ============================================================
def ensemble_predict(ensemble, X_scaled_tensor):
    preds = []
    with torch.no_grad():
        for model in ensemble:
            model.eval()
            p_scaled = model(X_scaled_tensor).squeeze(-1)
            p_raw = p_scaled * y_std + y_mean
            preds.append(p_raw)
    preds = torch.stack(preds, dim=0)
    return preds.mean(dim=0), preds.std(dim=0) + 1e-9

normal = torch.distributions.Normal(
    torch.tensor(0.0, device=device),
    torch.tensor(1.0, device=device)
)

def expected_improvement(mean, std, best_y, xi=0.01):
    imp = mean - best_y - xi
    Z = imp / std
    ei = imp * normal.cdf(Z) + std * torch.exp(normal.log_prob(Z))
    return torch.where(std > 0, ei, torch.zeros_like(ei))

def probability_of_improvement(mean, std, best_y, xi=0.0):
    imp = mean - best_y - xi
    Z = imp / std
    return normal.cdf(Z)

# ============================================================
# 8) WEEK 8 SETTINGS + LOCAL SEARCH
#    NOTE: decoding settings are kept for reporting consistency,
#          but selection is now ALWAYS greedy best-by-score.
# ============================================================
DECODING = {
    "temperature": 0.55,
    "top_p": 0.75,
    "top_k": 40,
    "max_tokens": 45000,  # n_candidates
}
CANDIDATE_MIX = {
    "local_frac": 0.80,
    "trust_radius": 0.14,
}
SCORE_WEIGHTS = {"w_mean": 0.70, "w_ei": 0.25, "w_pi": 0.05}

def clamp01(a):
    return np.minimum(1.0, np.maximum(0.0, a))

def is_duplicate(x, X_existing, tol=1e-6):
    return np.any(np.linalg.norm(X_existing - x, axis=1) < tol)

def propose_next_point_week8_greedy_best_by_score(
    ensemble, X_min, X_max, X_existing,
    best_x_raw,
    rr,
    xi,
    top_report=10,
    dup_tol=1e-6,
    decoding=DECODING,
    cand_mix=CANDIDATE_MIX,
    weights=SCORE_WEIGHTS
):
    n_candidates = int(decoding["max_tokens"])
    local_n = int(cand_mix["local_frac"] * n_candidates)
    global_n = n_candidates - local_n

    best_scaled = (best_x_raw - X_min) / (X_max - X_min + 1e-12)
    best_scaled = clamp01(best_scaled)

    r = float(cand_mix["trust_radius"])
    local = best_scaled + rr.normal(0.0, r, size=(local_n, 4)).astype(np.float32)
    local = clamp01(local)

    global_c = rr.random((global_n, 4), dtype=np.float32)
    candidates_scaled = np.vstack([local, global_c]).astype(np.float32)

    X_cand_tensor = torch.tensor(candidates_scaled, dtype=torch.float32, device=device)
    mean, std = ensemble_predict(ensemble, X_cand_tensor)

    best_y = float(np.max(y_train))
    ei = expected_improvement(mean, std, best_y, xi=xi)
    pi = probability_of_improvement(mean, std, best_y, xi=0.0)

    mean_np = mean.detach().cpu().numpy()
    std_np = std.detach().cpu().numpy()
    ei_np = ei.detach().cpu().numpy()
    pi_np = pi.detach().cpu().numpy()

    def norm01(v):
        v = np.asarray(v, dtype=np.float64)
        lo, hi = np.min(v), np.max(v)
        return (v - lo) / (hi - lo + 1e-12)

    mean_n = norm01(mean_np)
    ei_n = norm01(ei_np)
    pi_n = norm01(pi_np)

    score = (
        weights["w_mean"] * mean_n
        + weights["w_ei"] * ei_n
        + weights["w_pi"] * pi_n
    )

    # Non-duplicate indices
    keep = []
    for i in range(n_candidates):
        x_raw = X_min + candidates_scaled[i] * (X_max - X_min)
        if not is_duplicate(x_raw, X_existing, tol=dup_tol):
            keep.append(i)

    if len(keep) == 0:
        # fallback: best overall score (even if duplicates)
        chosen_i = int(np.argmax(score))
    else:
        keep = np.array(keep, dtype=int)
        # ALWAYS choose best-by-score (greedy)
        chosen_i = int(keep[np.argmax(score[keep])])

    def pack(j):
        xr = X_min + candidates_scaled[j] * (X_max - X_min)
        return {"idx": int(j), "x_raw": xr,
                "mean": float(mean_np[j]), "std": float(std_np[j]),
                "ei": float(ei_np[j]), "pi": float(pi_np[j]), "score": float(score[j])}

    chosen = pack(chosen_i)

    # Reporting: top by EI and top by SCORE among non-duplicates (or all if none)
    if len(keep) == 0:
        pool = np.arange(n_candidates)
    else:
        pool = keep

    pool_sorted_score = pool[np.argsort(score[pool])[::-1]]
    pool_sorted_ei = pool[np.argsort(ei_np[pool])[::-1]]

    report = {
        "top_by_ei": [pack(j) for j in pool_sorted_ei[:top_report]],
        "top_by_score": [pack(j) for j in pool_sorted_score[:top_report]],
    }

    meta = {
        "best_y": best_y,
        "n_candidates": n_candidates,
        "local_n": local_n,
        "global_n": global_n,
        "decoding": decoding,
        "cand_mix": cand_mix,
        "weights": weights,
        "selection": "GREEDY_BEST_BY_SCORE"
    }
    return chosen, report, meta

chosen, report, meta = propose_next_point_week8_greedy_best_by_score(
    ensemble=ensemble,
    X_min=X_min, X_max=X_max,
    X_existing=X_train,
    best_x_raw=current_best_x,
    rr=rng,
    xi=best_cfg["xi"],
    top_report=10,
    dup_tol=1e-6
)

next_x = chosen["x_raw"]
next_mean = chosen["mean"]
next_std = chosen["std"]
next_ei = chosen["ei"]
next_pi = chosen["pi"]
next_score = chosen["score"]

# ============================================================
# 9) Report (x_next in 6 decimals)
# ============================================================
def fmt_x6(x):
    return "[" + ", ".join(f"{v:.6f}" for v in x) + "]"

print("\n================ WEEK 8 FUNCTION 4 RESULTS (v3 — GREEDY BEST-BY-SCORE) ================")
print("Tuned surrogate config:", best_cfg)
print("Best CV-MSE (scaled y):", best_cv)

print("\nCURRENT BEST OBSERVED")
print("x_best =", fmt_x6(current_best_x), ", y_best =", f"{current_best_y:.6f}")

print("\nWEEK 8 SETTINGS")
print("Candidate mix:", meta["cand_mix"])
print("Decoding (kept for logging):", meta["decoding"])
print("Score weights:", meta["weights"])
print("Selection:", meta["selection"])
print("xi:", f"{best_cfg['xi']:.6f}")

print("\nTOP-10 NON-DUPLICATE CANDIDATES (ranked by EI)")
for i, r in enumerate(report["top_by_ei"], 1):
    print(f"{i:02d}) x={fmt_x6(r['x_raw'])} | mean={r['mean']:.6f} std={r['std']:.6f} EI={r['ei']:.6f} PI={r['pi']:.6f} SCORE={r['score']:.6f}")

print("\nTOP-10 NON-DUPLICATE CANDIDATES (ranked by SCORE used for selection)")
for i, r in enumerate(report["top_by_score"], 1):
    print(f"{i:02d}) x={fmt_x6(r['x_raw'])} | mean={r['mean']:.6f} std={r['std']:.6f} EI={r['ei']:.6f} PI={r['pi']:.6f} SCORE={r['score']:.6f}")

print("\nRECOMMENDED NEXT POINT (ALWAYS best-by-score; x_next in 6 decimals)")
print("x_next     =", fmt_x6(next_x))
print("mu(x_next) =", f"{next_mean:.6f}")
print("sigma      =", f"{next_std:.6f}")
print("EI         =", f"{next_ei:.6f}")
print("PI         =", f"{next_pi:.6f}")
print("SCORE      =", f"{next_score:.6f}")

delta = next_mean - current_best_y
print("\nREASONING")
print(f"- Local-maxima push: {int(CANDIDATE_MIX['local_frac']*100)}% candidates sampled near x_best (trust_radius={CANDIDATE_MIX['trust_radius']}).")
print("- Selection is now deterministic greedy: ALWAYS chooses the highest SCORE among non-duplicate candidates.")
print(f"- Predicted Δmean vs best: {delta:.6f} (positive means surrogate expects improvement).")
print(f"- PI={next_pi:.6f} (~{next_pi*100:.2f}% chance to beat best, under the surrogate).")
print("- Duplicate avoidance prevents re-querying an existing point.")


Current best index: 36
Current best X: [0.497045 0.450388 0.380113 0.297612]
Current best y: -2.5615259470796796

=== Hyperparameter tuning (random search + 4-fold CV) ===
trial 01/35  cv_mse=0.394122  cfg={'hidden': (32, 32), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 1e-05, 'epochs': 900, 'n_ensemble': 9, 'xi': 0.0}
trial 02/35  cv_mse=0.465522  cfg={'hidden': (64, 64), 'dropout': 0.0, 'lr': 0.0001, 'weight_decay': 1e-05, 'epochs': 1200, 'n_ensemble': 9, 'xi': 0.01}
trial 03/35  cv_mse=0.313793  cfg={'hidden': (64, 64), 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0, 'epochs': 1200, 'n_ensemble': 7, 'xi': 0.005}
trial 04/35  cv_mse=0.170939  cfg={'hidden': (64, 32), 'dropout': 0.0, 'lr': 0.003, 'weight_decay': 0.0001, 'epochs': 900, 'n_ensemble': 7, 'xi': 0.01}
trial 05/35  cv_mse=0.350084  cfg={'hidden': (64, 64), 'dropout': 0.05, 'lr': 0.0003, 'weight_decay': 1e-06, 'epochs': 700, 'n_ensemble': 9, 'xi': 0.01}
trial 06/35  cv_mse=0.390148  cfg={'hidden': (32, 32), 'dropout': 0.2